# Browse every spaCR workflow

[Standard notebook](https://github.com/EinarOlafsson/spacr/blob/nightly/Notebooks/00_workflow_browser.ipynb) · [Help](https://github.com/EinarOlafsson/spacr/blob/nightly/Notebooks/README.md) · [All workflows](https://github.com/EinarOlafsson/spacr/blob/nightly/Notebooks/README.md)

[Setup](#setup) · [Data](#data) · [Inputs](#inputs) · [Advanced](#advanced) · [Run](#run) · [Outputs](#outputs)

## Setup

In [ ]:
#@title 1. Setup (check Install only when needed)
INSTALL_SPACR = False #@param {"type":"boolean"}
PACKAGE = "spacr" #@param {"type":"string"}
if INSTALL_SPACR:
    import subprocess, sys
    if PACKAGE != "spacr" and not (PACKAGE.startswith("spacr==") and all(c.isdigit() or c == "." for c in PACKAGE[7:])):
        raise ValueError("Use spacr or spacr==<published version>.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", PACKAGE, "ipywidgets"])
from importlib.metadata import version
print("spaCR", version("spacr"))
from google.colab import output
output.enable_custom_widget_manager()


In [ ]:
#@title 2. Load matching notebook controls
from pathlib import Path
from urllib.request import urlopen
import hashlib
helper = Path("_spacr_notebook_widgets.py")
expected = 'b3cee160648607b49ef051c2424c06a28631a4a11a7b52d1063f73cec643f7c0'
if not helper.is_file() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
    payload = urlopen("https://raw.githubusercontent.com/EinarOlafsson/spacr/nightly/Notebooks/_spacr_notebook_widgets.py", timeout=30).read()
    if hashlib.sha256(payload).hexdigest() != expected:
        raise RuntimeError("Notebook/helper versions differ. Reopen the latest notebook before continuing.")
    helper.write_bytes(payload)


## Data

In [ ]:
#@title 3. Data (Drive/upload only when selected)
DATA_SOURCE = "Existing runtime path" #@param ["Existing runtime path", "Google Drive", "Upload files"]
DATA_PATH = "/content/data" #@param {"type":"string"}
if DATA_SOURCE == "Google Drive":
    from google.colab import drive
    drive.mount("/content/drive")
elif DATA_SOURCE == "Upload files":
    from google.colab import files
    from pathlib import Path
    destination = Path(DATA_PATH)
    destination.mkdir(parents=True, exist_ok=True)
    for filename, payload in files.upload().items():
        (destination / Path(filename).name).write_bytes(payload)
print("Set workflow paths below to your data location:", DATA_PATH)


## Inputs

In [ ]:
#@title Picker
from spacr.version import version_str
from spacr.cli import MODULES, resolve_module, resolve_settings
from IPython.display import Markdown, display

print("spaCR", version_str)
module_key = "measure" #@param ["activation", "align", "analyze_plaques", "anndata_export", "barcode_qc", "cellpose_masks", "classify", "classify_merged", "convert", "endodyogeny", "explain_cv", "external_masks", "foreign", "host_pathogen", "illumination", "invasion", "investigate_hit", "map_barcodes", "mask", "measure", "ml_analyze", "motility", "ops", "recruitment", "regression", "replication", "simulation", "spatial_map", "spatial_masks", "timelapse", "train_cellpose", "train_only", "umap"]
module_picker = None
try:
    import ipywidgets as widgets
except ImportError:
    print("Optional picker unavailable; edit module_key above.")
else:
    module_picker = widgets.Dropdown(
        options=[(f"{key}: {row.summary}", key) for key, row in sorted(MODULES.items())],
        value=module_key, description="Workflow:", layout=widgets.Layout(width="100%"))
    display(module_picker)


In [ ]:
#@title Settings
module = resolve_module(module_picker.value if module_picker is not None else module_key)
settings_file = None #@param {"type":"raw"}
overrides = [] #@param {"type":"raw"}
settings = resolve_settings(module, settings_file, overrides)
dotted = module.entry.replace(":", ".")
api_url = "https://einarolafsson.github.io/spacr/api/" + module.module_name.replace(".", "/") + "/index.html#" + dotted
display(Markdown(f"### {module.key}: {module.summary}\n\n[Public API]({api_url})"))
print("Required inputs:", *module.requires, sep="\n- ")
print("Writes:", *module.writes, sep="\n- ")
print(f"{len(settings)} current settings loaded. Edit overrides above or settings below.")


## Advanced

In [ ]:
#@title Editor
if module_picker is not None:
    try:
        from _spacr_notebook_widgets import show_workflow_settings_editor
    except ImportError:
        print("Optional editor helper unavailable; edit settings or overrides in Python.")
    else:
        advanced, editor, apply_button = show_workflow_settings_editor(settings, module)
else:
    print("Edit settings directly, or edit overrides and rerun the configuration cell.")


## Run

In [ ]:
#@title Run
RUN_WORKFLOW = False #@param {"type":"boolean"}
if RUN_WORKFLOW:
    from importlib import import_module
    function = getattr(import_module(module.module_name), module.func_name)
    result = function(settings if module.call_style == "settings" else settings["src"])
else:
    print("No analysis started. Configure inputs and enable RUN_WORKFLOW when ready.")


## Outputs

In [ ]:
#@title 9. Inspect outputs (bounded list; preview only on click)
PREVIEW_OUTPUTS = False #@param {"type":"boolean"}
OUTPUT_FOLDER = "/content/data" #@param {"type":"string"}
if PREVIEW_OUTPUTS:
    from _spacr_notebook_widgets import preview_outputs
    output_picker, preview_button, output_display = preview_outputs(OUTPUT_FOLDER)


In [ ]:
#@title 10. Download one selected output (optional)
DOWNLOAD_FILE = False #@param {"type":"boolean"}
FILE_PATH = "" #@param {"type":"string"}
if DOWNLOAD_FILE:
    from pathlib import Path
    path = Path(FILE_PATH).expanduser()
    if not path.is_file():
        raise ValueError("Choose an existing output file.")
    from google.colab import files
    files.download(str(path))
